In [17]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. The Neural Network Blueprint
class PolicyNetwork(nn.Module):
    def __init__(self, input_size, hidden_size, num_actions):
        super(PolicyNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.layer2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, num_actions)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        return self.output_layer(x) 

# 2. The Grid Car Game Environment Blueprint
class GridCarEnv:
    def __init__(self, width=5, height=5):
        self.width = width
        self.height = height
        self.goal = [width-1, height-1] # Goal is the top right corner
        self.state = [0, 0] # Start at bottom left
        self.step_count = 0

    def reset(self):
        self.state = [0, 0]
        self.step_count = 0
        return self.state

    def step(self, action):
        self.step_count += 1
        x, y = self.state
        
        # 0: Up, 1: Right, 2: Down, 3: Left
        if action == 0: y += 1
        elif action == 1: x += 1
        elif action == 2: y -= 1
        elif action == 3: x -= 1

        # Keep the car inside the grid bounds
        x = max(0, min(x, self.width - 1))
        y = max(0, min(y, self.height - 1))
        self.state = [x, y]

        # Check if we reached the goal or took too many steps
        if self.state == self.goal:
            return self.state, 10.0, True # Big reward for winning!
        elif self.step_count > 50:
            return self.state, -1.0, True # Cut the game off if taking too long
        else:
            return self.state, -0.1, False # Tiny penalty to encourage speed

# 3. Trajectory Generator
def generate_trajectory(network, env):
    trajectory = []
    state = env.reset() 
    done = False
    
    while not done:
        state_tensor = torch.FloatTensor(state)
        action_scores = network(state_tensor)
        
        action_probs = torch.softmax(action_scores, dim=0)
        action_distribution = torch.distributions.Categorical(action_probs)
        action = action_distribution.sample()
        
        next_state, reward, done = env.step(action.item())
        
        trajectory.append({
            'state': state,
            'action': action,
            'reward': reward
        })
        state = next_state
        
    return trajectory

# 4. The Trainer (Slightly improved to batch updates)
def train_reinforce(network, optimizer, trajectories):
    optimizer.zero_grad() # Clear old math
    total_loss = 0
    
    for trajectory in trajectories:
        total_reward = sum([step['reward'] for step in trajectory])
        
        for step in trajectory:
            state_tensor = torch.FloatTensor(step['state'])
            action_taken = step['action']
            
            action_scores = network(state_tensor)
            action_probs = torch.softmax(action_scores, dim=0)
            
            log_prob = torch.log(action_probs[action_taken] + 1e-8) # 1e-8 prevents math errors
            total_loss -= log_prob * total_reward 
            
    total_loss.backward()  # Calculate how to change the network
    optimizer.step()       # Actually change the network

# 5. The Evaluator
def compare_to_exact_solution(network, exact_solution_table, grid_width, grid_height):
    correct_moves = 0
    total_squares = grid_width * grid_height
    
    for x in range(grid_width):
        for y in range(grid_height):
            state = [x, y]
            
            state_tensor = torch.FloatTensor(state)
            action_scores = network(state_tensor)
            network_action = torch.argmax(action_scores).item() 
            
            perfect_action = exact_solution_table[x][y]
            
            if network_action == perfect_action:
                correct_moves += 1
                
    accuracy = (correct_moves / total_squares) * 100
    print(f"Network is {accuracy:.2f}% identical to the perfect mathematical solution!")

In [18]:
# 1. Create the Environment (5x5 grid)
grid_size = 5
env = GridCarEnv(width=grid_size, height=grid_size)

# 2. Create the Network (Input is [x,y] so size is 2, Output is 4 actions)
net = PolicyNetwork(input_size=2, hidden_size=64, num_actions=4)

# 3. Create the Optimizer
optimizer = optim.Adam(net.parameters(), lr=0.01)

# 4. Create a dummy Exact Solution Table
# For a 5x5 grid going from [0,0] to [4,4], going Right (1) or Up (0) is always correct.
# We will just fill it with 1s (Right) and 0s (Up) to represent a perfect path.
exact_solution_table = []
for x in range(grid_size):
    row = []
    for y in range(grid_size):
        # If we are at the top edge, we must go Right (1)
        if y == grid_size - 1: row.append(1)
        # Otherwise, going Up (0) is a safe perfect move
        else: row.append(0)
    exact_solution_table.append(row)

In [ ]:
# --- BLOCK 5: The Main Execution Loop ---

# NOTE: You will need to define 'env' (your grid game) 
# and 'exact_solution_table' before this block will run!

# Let's pretend we want to train for 500 "epochs" (training cycles)
num_epochs = 500
games_per_epoch = 10 # Play 10 games before updating the brain

print("Starting training...")

for epoch in range(num_epochs):
    
    # 1. Generate a batch of trajectories (play the game)
    batch_trajectories = []
    for _ in range(games_per_epoch):
        # We use 'net' from Block 1, and 'env' (which you need to build)
        trajectory = generate_trajectory(net, env) 
        batch_trajectories.append(trajectory)
        
    # 2. Train the network on those games using REINFORCE
    # We use 'optimizer' from Block 1
    train_reinforce(net, optimizer, batch_trajectories)
    
    # Print an update every 100 epochs so we know it's working
    if epoch % 100 == 0:
        print(f"Finished Epoch {epoch}")

print("Training complete!")

# 3. Check our work against the exact mathematical solution!
# (Assuming a 10x10 grid for this example)
compare_to_exact_solution(net, exact_solution_table, grid_width=grid_size, grid_height=grid_size)

Starting training...
Finished Epoch 0


In [ ]:
# Test the trained network against the perfect math solution
# Assuming a 5x5 grid for this example
compare_to_exact_solution(net, exact_solution_table, grid_width=grid_size, grid_height=grid_size)